# MMLab: Processing and Visualization
![Logo MMLab](../../Assets/Figures/Logo%20MMLab.png)


In [ ]:
import pandas as pd
import numpy as np
import glob
import os
from scipy import signal
from sklearn.metrics import mutual_info_score
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Your existing functions
def compute_coherence(x, y, fs):
    f, Cxy = signal.coherence(x, y, fs)
    return f, Cxy

def crosscorr(datax, datay, lag=0):
    """ Lag-N cross correlation. Shift datax by N elements. """
    return datax.corr(datay.shift(lag))

def compute_mutual_information(x: np.ndarray, y: np.ndarray) -> float:
    """
    Compute mutual information between two time series using KDE.
    """
    # Clean data
    mask = ~np.isnan(x) & ~np.isnan(y)
    x = x[mask]
    y = y[mask]
    
    if len(x) < 2 or len(y) < 2:
        return 0.0
    
    # Standardize the data
    x = (x - np.mean(x)) / np.std(x)
    y = (y - np.mean(y)) / np.std(y)
    
    # Create KDE estimators
    kde_joint = stats.gaussian_kde(np.vstack([x, y]))
    kde_x = stats.gaussian_kde(x)
    kde_y = stats.gaussian_kde(y)
    
    # Sample points for numerical integration
    n_samples = 50
    x_range = np.linspace(min(x) - 1, max(x) + 1, n_samples)
    y_range = np.linspace(min(y) - 1, max(y) + 1, n_samples)
    X, Y = np.meshgrid(x_range, y_range)
    positions = np.vstack([X.ravel(), Y.ravel()])
    
    # Evaluate densities
    joint_density = kde_joint(positions).reshape(X.shape)
    x_density = kde_x(X[0,:])
    y_density = kde_y(Y[:,0])
    X_density, Y_density = np.meshgrid(x_density, y_density)
    
    # Compute MI
    with np.errstate(divide='ignore', invalid='ignore'):
        mi_density = joint_density * np.log(joint_density / (X_density * Y_density))
    mi = np.nansum(mi_density) * (x_range[1] - x_range[0]) * (y_range[1] - y_range[0])
    
    return max(0, mi)  # Ensure non-negative MI

def compute_coupling_statistics(name, motion_ts, sound_ts, time):
    """Your existing coupling statistics function"""
    # Ensure inputs are numpy arrays
    motion_ts = np.array(motion_ts)
    sound_ts = np.array(sound_ts)
    time = np.array(time)

    # Check if inputs are scalar (single values)
    if motion_ts.ndim == 0 or sound_ts.ndim == 0 or time.ndim == 0:
        return pd.DataFrame({
            'scene': [name],
            'lags': [np.nan],
            'crosscorr': [np.nan],
        }), np.nan, np.nan, np.nan, np.nan, np.nan, np.nan

    # Check if inputs have the same length
    if len(motion_ts) != len(sound_ts) or len(motion_ts) != len(time):
        raise ValueError("motion_ts, sound_ts, and time must have the same length")

    # normalize and center the data
    motion_ts = (motion_ts - np.min(motion_ts)) / (np.max(motion_ts) - np.min(motion_ts))
    motion_ts = motion_ts - np.mean(motion_ts)
    sound_ts = (sound_ts - np.min(sound_ts)) / (np.max(sound_ts) - np.min(sound_ts))
    sound_ts = sound_ts - np.mean(sound_ts)

    # check if values are finite
    if not np.all(np.isfinite(motion_ts)) or not np.all(np.isfinite(sound_ts)):
        return pd.DataFrame({
            'scene': [name],
            'lags': [np.nan],
            'crosscorr': [np.nan],
        }), np.nan, np.nan, np.nan, np.nan, np.nan, np.nan

    # Compute sampling frequency
    fs = 1/np.mean(np.diff(time))

    # compute the average mutual information
    mi = []
    lags = [-0.3, -0.2, -0.1, 0, 0.1, 0.2, 0.3]
    fs = 1/np.mean(np.diff(time))
    lags_samples = [int(lag * fs) for lag in lags]
    
    for lag in lags_samples:
        if lag < 0:
            motsub = motion_ts[:lag]
            soundsub = sound_ts[-lag:]
        elif lag > 0:
            motsub = motion_ts[lag:]
            soundsub = sound_ts[:-lag]
        else:
            motsub = motion_ts
            soundsub = sound_ts
            
        # Add smoothing to capture temporal dependencies
        window = int(0.1 * fs)  # 100ms window
        if window > 1:
            motsub = np.convolve(motsub, np.ones(window)/window, mode='valid')
            soundsub = np.convolve(soundsub, np.ones(window)/window, mode='valid')
        
        mi_value = compute_mutual_information(motsub, soundsub)
        mi.append(mi_value)
    
    max_mi = np.max(mi)
    optimal_lag = lags[np.argmax(mi)]
    
    #################################################### Coherence
    # Compute coherence
    f, Cxy = compute_coherence(motion_ts, sound_ts, fs)

    # keep all values lower than 10 Hz
    mask = f < 10
    Cxy = Cxy[mask]
    f = f[mask]

    # maximum coherence
    coherence = np.max(Cxy)

    # Frequency of max coherence
    f_max = f[np.argmax(Cxy)]
        
    #################################################### Cross-correlation
    lag_seconds = 0.3
    lag_points = int(lag_seconds * fs)

    # Create a range of lags to examine
    lags = np.arange(-lag_points, lag_points + 1)

    # calc cross-cor
    cc = [crosscorr(pd.Series(motion_ts), pd.Series(sound_ts), lag) for lag in lags]

    # Calculate cross-correlation
    crosscorrdf = pd.DataFrame({
        'scene': name,
        'lags': lags*(1/fs),
        'crosscorr': cc,
    })

    ########## Phase locking value
    # Compute analytic signal (using Hilbert transform)
    motion_analytic = signal.hilbert(motion_ts)
    sound_analytic = signal.hilbert(sound_ts)

    # Extract instantaneous phase
    motion_phase = np.angle(motion_analytic)
    sound_phase = np.angle(sound_analytic)

    # Compute phase difference
    phase_diff = motion_phase - sound_phase

    # compute the plv
    plv = np.abs(np.mean(np.exp(1j * phase_diff)))
    
    # make phase_diff a regular list not a numpy array
    try:
        motion_phase = motion_phase.tolist()[0] if hasattr(motion_phase, 'tolist') else motion_phase[0]
        phase_diff = phase_diff.tolist()[0] if hasattr(phase_diff, 'tolist') else phase_diff[0]
    except:
        motion_phase = np.mean(motion_phase)
        phase_diff = np.mean(phase_diff)

    return crosscorrdf, coherence, f_max, max_mi, phase_diff, plv, optimal_lag

def calculate_p1_p2_coupling_stats(merged_folder='../merged_filteredtimeseries/', output_file='p1_p2_coupling_statistics.csv'):
    """
    Loop through merged time series files and calculate coupling statistics between P1 and P2
    """
    
    # Find all merged CSV files
    csv_files = glob.glob(os.path.join(merged_folder, "*.csv"))
    print(f"Found {len(csv_files)} files to process")
    
    # Define P1-P2 variable pairs to analyze
    variable_pairs = [
        # Audio Envelope
        ('Amplitude_Envelope_P1', 'Amplitude_Envelope_P2'),
        
        # Heart Rate
        ('heart_rate_P1', 'heart_rate_P2'),
        
        # Respiration
        ('Filtered_Respiration_P1', 'Filtered_Respiration_P2'),
        
        # EMG - Bicep
        ('Filtered_EMG_Bicep_P1', 'Filtered_EMG_Bicep_P2'),
        
        # EMG - Tricep
        ('Filtered_EMG_Tricep_P1', 'Filtered_EMG_Tricep_P2'),
        
        # Motion Tracking (if available)
        ('right_index_x_P1', 'right_index_x_P2'),
        ('right_index_y_P1', 'right_index_y_P2'),
        ('right_index_z_P1', 'right_index_z_P2'),
        
        # Cross-modal pairs (examples)
        ('heart_rate_P1', 'heart_rate_P2'),
        ('Amplitude_Envelope_P1', 'heart_rate_P1'),  # Within P1
        ('Amplitude_Envelope_P2', 'heart_rate_P2'),  # Within P2
        ('Amplitude_Envelope_P1', 'heart_rate_P2'),  # Cross-participant
        ('heart_rate_P1', 'Amplitude_Envelope_P2'),  # Cross-participant
        
        # Respiration-Heart Rate coupling
        ('Filtered_Respiration_P1', 'heart_rate_P1'),  # Within P1
        ('Filtered_Respiration_P2', 'heart_rate_P2'),  # Within P2
        ('Filtered_Respiration_P1', 'heart_rate_P2'),  # Cross-participant
        ('heart_rate_P1', 'Filtered_Respiration_P2'),  # Cross-participant
    ]
    
    # Initialize results list
    results = []
    
    # Loop through each file
    for file_path in csv_files:
        print(f"\nProcessing: {os.path.basename(file_path)}")
        
        try:
            # Load the data
            df = pd.read_csv(file_path)
            
            # Extract metadata from filename and dataframe
            filename = os.path.basename(file_path)
            
            # Try to extract condition information
            condition_vision = df['ConditionVision'].iloc[0] if 'ConditionVision' in df.columns else 'Unknown'
            condition_movement = df['ConditionMovement'].iloc[0] if 'ConditionMovement' in df.columns else 'Unknown'
            trial = df['Trial'].iloc[0] if 'Trial' in df.columns else 'Unknown'
            
            # Get time vector
            time = df['Time'].values
            
            # Check if we have enough data (at least 3 seconds)
            if time.max() - time.min() < 3.0:
                print(f"  Skipping {filename}: insufficient data duration ({time.max() - time.min():.2f}s)")
                continue
            
            # Set up sliding window analysis
            window_duration = 3.0  # seconds
            step_size = 1.0  # seconds
            n_windows = int((time.max() - time.min() - window_duration) / step_size) + 1
            
            print(f"  Analyzing {n_windows} windows of {window_duration}s each")
            
            # Loop through each variable pair
            for var1, var2 in variable_pairs:
                if var1 not in df.columns or var2 not in df.columns:
                    continue
                    
                print(f"    Processing {var1} vs {var2}")
                
                # Loop through sliding windows
                for window_idx in range(n_windows):
                    window_start = time.min() + window_idx * step_size
                    window_end = window_start + window_duration
                    
                    # Extract window data
                    window_mask = (time >= window_start) & (time <= window_end)
                    time_window = time[window_mask]
                    var1_window = df[var1].values[window_mask]
                    var2_window = df[var2].values[window_mask]
                    
                    # Skip if not enough data in window or too many NaNs
                    if len(time_window) < 100 or np.sum(~np.isnan(var1_window)) < 50 or np.sum(~np.isnan(var2_window)) < 50:
                        continue
                    
                    try:
                        # Calculate coupling statistics
                        crosscorrdf, coherence, f_max, max_mi, phase_diff, plv, optimal_lag = \
                            compute_coupling_statistics(filename, var1_window, var2_window, time_window)
                        
                        # Extract max cross-correlation
                        max_crosscorr = crosscorrdf['crosscorr'].abs().max() if not crosscorrdf.empty else np.nan
                        lag_at_max_crosscorr = crosscorrdf.loc[crosscorrdf['crosscorr'].abs().idxmax(), 'lags'] if not crosscorrdf.empty else np.nan
                        
                        # Store results
                        result = {
                            'filename': filename,
                            'condition_vision': condition_vision,
                            'condition_movement': condition_movement,
                            'trial': trial,
                            'window_idx': window_idx,
                            'window_start': window_start,
                            'window_end': window_end,
                            'var1': var1,
                            'var2': var2,
                            'variable_pair_type': classify_variable_pair(var1, var2),
                            'max_crosscorr': max_crosscorr,
                            'lag_at_max_crosscorr': lag_at_max_crosscorr,
                            'max_coherence': coherence,
                            'freq_at_max_coherence': f_max,
                            'max_mutual_info': max_mi,
                            'optimal_lag_mi': optimal_lag,
                            'phase_locking_value': plv,
                            'mean_phase_diff': phase_diff,
                            'n_samples': len(time_window),
                            'sampling_rate': 1/np.mean(np.diff(time_window))
                        }
                        
                        results.append(result)
                        
                    except Exception as e:
                        print(f"      Error in window {window_idx}: {e}")
                        continue
                        
        except Exception as e:
            print(f"  Error processing {filename}: {e}")
            continue
    
    # Convert to DataFrame
    if results:
        results_df = pd.DataFrame(results)
        
        # Save to CSV
        results_df.to_csv(output_file, index=False)
        print(f"\n✓ Saved {len(results_df)} coupling statistics to {output_file}")
        
        # Print summary statistics
        print(f"\nSummary:")
        print(f"  Files processed: {results_df['filename'].nunique()}")
        print(f"  Variable pairs analyzed: {results_df['variable_pair_type'].nunique()}")
        print(f"  Total windows analyzed: {len(results_df)}")
        print(f"  Conditions: {results_df['condition_vision'].unique()} x {results_df['condition_movement'].unique()}")
        
        return results_df
    else:
        print("No results generated - check your data files and variable names")
        return pd.DataFrame()

def classify_variable_pair(var1, var2):
    """
    Classify the type of variable pair for easier analysis
    """
    # Extract modality information
    modalities = []
    participants = []
    
    for var in [var1, var2]:
        if 'Amplitude_Envelope' in var:
            modalities.append('Audio')
        elif 'heart_rate' in var:
            modalities.append('ECG')
        elif 'Respiration' in var:
            modalities.append('Respiration')
        elif 'EMG' in var:
            modalities.append('EMG')
        elif 'right_index' in var:
            modalities.append('Motion')
        else:
            modalities.append('Other')
            
        if '_P1' in var:
            participants.append('P1')
        elif '_P2' in var:
            participants.append('P2')
        else:
            participants.append('Unknown')
    
    # Classify pair type
    if participants[0] == participants[1]:
        if participants[0] == 'P1':
            pair_type = 'Within_P1'
        else:
            pair_type = 'Within_P2'
    else:
        pair_type = 'Between_Participants'
    
    if modalities[0] == modalities[1]:
        modality_type = f"Same_{modalities[0]}"
    else:
        modality_type = f"Cross_{modalities[0]}_{modalities[1]}"
    
    return f"{pair_type}_{modality_type}"

def create_summary_plots(results_df, output_folder='coupling_plots/'):
    """
    Create summary plots of the coupling statistics
    """
    import matplotlib.pyplot as plt
    
    os.makedirs(output_folder, exist_ok=True)
    
    # Set style
    plt.style.use('ggplot')
    
    # 1. Cross-correlation by condition and variable pair type
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.boxplot(data=results_df, x='condition_vision', y='max_crosscorr', 
                hue='variable_pair_type', ax=ax)
    plt.xticks(rotation=45)
    plt.title('Maximum Cross-Correlation by Condition and Variable Pair Type')
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, 'crosscorr_by_condition.png'), dpi=300, bbox_inches='tight')
    plt.close()
    
    # 2. Phase Locking Value comparison
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.boxplot(data=results_df, x='condition_movement', y='phase_locking_value', 
                hue='variable_pair_type', ax=ax)
    plt.xticks(rotation=45)
    plt.title('Phase Locking Value by Movement Condition and Variable Pair Type')
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, 'plv_by_condition.png'), dpi=300, bbox_inches='tight')
    plt.close()
    
    # 3. Mutual Information heatmap
    pivot_mi = results_df.groupby(['condition_vision', 'condition_movement', 'variable_pair_type'])['max_mutual_info'].mean().unstack()
    
    fig, ax = plt.subplots(figsize=(14, 10))
    sns.heatmap(pivot_mi, annot=True, fmt='.3f', cmap='viridis', ax=ax)
    plt.title('Average Mutual Information by Condition and Variable Pair Type')
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, 'mi_heatmap.png'), dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Summary plots saved to {output_folder}")


In [ ]:
merged_folder = '../4a/Processed/merged_filteredtimeseries/'  # Update this path
output_file = './p1_p2_coupling_statistics.csv'

# Calculate coupling statistics
results_df = calculate_p1_p2_coupling_stats(merged_folder, output_file)

# Create summary plots if results were generated
if not results_df.empty:
    create_summary_plots(results_df, output_folder='coupling_plots/')
    
    # Display some example results
    print("\nExample results:")
    print(results_df.head())
    
    print("\nVariable pair types found:")
    print(results_df['variable_pair_type'].value_counts())